# cellmap-flow on Colab (production dashboard, cloudflared for NG + inference)

Runs the real `cellmap_flow_app` (dashboard Flask) and
`cellmap_flow_server` (inference Flask) in this Colab session.

URL split (the only way to make the production dashboard's NG iframe
actually load in Colab):

| component  | reachable via            | why                                    |
| ---------- | ------------------------ | -------------------------------------- |
| dashboard  | Colab `proxyPort`        | auth-gated, top-level tab, SSE-capable |
| NG iframe  | cloudflared quick tunnel | public, no cookie needed in iframe     |
| inference  | cloudflared quick tunnel | public, no cookie needed in iframe     |

If NG were on `proxyPort` too, Chrome's 3rd-party-cookie block would
prevent the dashboard's iframe from authenticating to Colab's proxy
and NG would never load. Same for inference chunks. Moving both onto
public trycloudflare URLs sidesteps the cookie problem entirely.

The trycloudflare URLs are public but die when this Colab session
dies (a few hours). Fine for a personal demo; don't post them.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**.
2. **Run all cells.** The install cell may restart the kernel automatically
   (Colab pre-loads numpy at startup; pip-installing a different
   version requires a restart). If you see the kernel restart,
   **click Run All again**.
3. Click the printed Dashboard URL.
4. The dashboard's NG iframe loads with raw + inference layers
   pre-configured. Adjust Input / Postprocess and click Submit All —
   changes flow through the production `/api/process` route in the
   same kernel.

## 1. Install

In [ ]:
# Let pip pick a consistent numpy/scipy/skimage set — the kernel
# auto-restart below handles any version change vs. Colab preinstalled.
%pip install -q "cellmap-flow[bioimageio] @ git+https://github.com/janelia-cellmap/cellmap-flow.git@browser-inference" huggingface_hub s3fs
%pip install -q --force-reinstall "bioimageio.core==0.9.6" "bioimageio.spec==0.5.7.4"

# cloudflared binary — used to mint public trycloudflare URLs for the
# NG + inference servers (see top-level notebook docstring for why).
import os, subprocess
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("[install] downloading cloudflared ...")
    subprocess.check_call([
        "wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", "/usr/local/bin/cloudflared",
    ])
    subprocess.check_call(["chmod", "+x", "/usr/local/bin/cloudflared"])
    print("[install] cloudflared installed.")
else:
    print("[install] cloudflared already present.")

# Pip mutates packages on disk but Colab pre-imports numpy at kernel
# startup. If the kernel's loaded numpy doesn't match what's now on
# disk, the next `import cellmap_flow.globals` will ImportError.
# Detect that here and force-restart the kernel — Colab auto-spawns a
# fresh kernel. The user just clicks "Run all" again; cell 1 sees
# packages already installed (fast no-op) and the check passes.
try:
    import cellmap_flow.globals  # forces the same import chain as cell 5
    print("[install] kernel is in sync with on-disk packages, ready to proceed.")
except Exception as e:
    # numpy / scipy version skew typically shows up as ImportError or
    # AttributeError (the partially-loaded module is there but missing
    # a symbol). Either way: kernel needs to restart.
    print(f"[install] kernel state mismatched on-disk packages ({e.__class__.__name__}: {e}).")
    print("[install] restarting kernel in 3s — when it comes back, run all cells again.")
    import time, os
    time.sleep(3)
    os.kill(os.getpid(), 9)

## 2. Configure model + dataset

`MODEL_TYPE = "huggingface"` for a cellmap HF model, or `"bioimage"`
for a BMZ model.

T4 sizing notes:
- 178³ HF models (`fly_organelles_run07_*`): fit cleanly.
- 288³ HF models (`jrc_mus-livers_*`): borderline, OOM-prone.
- 2D BMZ models (`hiding-blowfish`): trivial, fits anywhere.


In [ ]:
MODEL_TYPE = "bioimage"      # or "huggingface"

# --- Mode A: huggingface ---
HF_REPO = "cellmap/fly_organelles_run07_432000"
HF_NAME = HF_REPO.split("/")[-1]
HF_DATASET = (
    "https://janelia-cosem-datasets.s3.amazonaws.com/"
    "jrc_mus-liver/jrc_mus-liver.zarr/recon-1/em/fibsem-uint8"
)

# --- Mode B: bioimage (BMZ) ---
BMZ_MODEL = "hiding-blowfish"
BMZ_VOXEL_SIZE = "8,8,8"
BMZ_DATASET = (
    "https://janelia-cosem-datasets.s3.amazonaws.com/"
    "jrc_hela-2/jrc_hela-2.zarr/recon-1/em/fibsem-uint8/s1"
)

INFERENCE_PORT = 8765
NG_PORT = 9090           # neuroglancer Tornado server
DASHBOARD_PORT = 8501

if MODEL_TYPE == "huggingface":
    MODEL_NAME = HF_NAME
    DATASET = HF_DATASET
elif MODEL_TYPE == "bioimage":
    MODEL_NAME = BMZ_MODEL
    DATASET = BMZ_DATASET
else:
    raise ValueError(f"unknown MODEL_TYPE={MODEL_TYPE!r}")

print(f"MODEL_TYPE = {MODEL_TYPE}")
print(f"MODEL_NAME = {MODEL_NAME}")
print(f"DATASET    = {DATASET}")

## 3. Start the cellmap-flow inference server (subprocess)

In [ ]:
import os, subprocess, time

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if MODEL_TYPE == "huggingface":
    cmd = [
        "cellmap_flow_server", "huggingface",
        "--repo", HF_REPO, "--name", HF_NAME,
        "-d", HF_DATASET, "--port", str(INFERENCE_PORT),
    ]
elif MODEL_TYPE == "bioimage":
    cmd = [
        "cellmap_flow_server", "bioimage",
        "--model-name", BMZ_MODEL, "--voxel-size", BMZ_VOXEL_SIZE,
        "--name", BMZ_MODEL,
        "-d", BMZ_DATASET, "--port", str(INFERENCE_PORT),
    ]
print("starting:", " ".join(cmd))
server = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    env={**os.environ},
)
print(f"server pid={server.pid}, waiting for it to listen on :{INFERENCE_PORT} ...")
for _ in range(300):
    line = server.stdout.readline()
    if not line: time.sleep(0.25); continue
    print(line, end="")
    if "Running on" in line or f":{INFERENCE_PORT}" in line:
        print("\n[server] ready.")
        break


## 4. Tunnel the inference server with cloudflared

Mints a `https://<random>.trycloudflare.com` URL pointing at the
local inference port. NG (running in the dashboard iframe) will
fetch chunks from this URL. Public-but-ephemeral — dies with the
Colab session.

In [ ]:
import subprocess, re, time

def start_cloudflared(port: int, label: str, timeout: float = 30) -> tuple[str, subprocess.Popen]:
    """Spawn `cloudflared tunnel --url http://localhost:<port>` and return
    (public_url, process). Caller is responsible for keeping the process
    alive — tunnel dies when the process exits."""
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://localhost:{port}",
         "--no-autoupdate", "--metrics", "127.0.0.1:0"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    pat = re.compile(r"https://[-a-z0-9]+\.trycloudflare\.com")
    deadline = time.time() + timeout
    while time.time() < deadline:
        line = proc.stdout.readline()
        if not line:
            if proc.poll() is not None:
                raise RuntimeError(f"cloudflared exited early (rc={proc.returncode})")
            time.sleep(0.1); continue
        # cloudflared logs are noisy; surface only the URL-bearing line.
        m = pat.search(line)
        if m:
            url = m.group(0)
            print(f"[{label}] {url}")
            return url, proc
    proc.terminate()
    raise RuntimeError(f"cloudflared did not emit URL within {timeout}s for port {port}")

INFERENCE_URL, _inference_tunnel = start_cloudflared(INFERENCE_PORT, "inference")
print(f"INFERENCE_URL = {INFERENCE_URL}")

## 5. Tunnel Neuroglancer + start the dashboard

Binds NG's Tornado server to a known port, opens a cloudflared
tunnel for it (so the dashboard iframe can load it cross-origin
without needing Colab's session cookie), then pre-populates the
dashboard state and runs the dashboard Flask in a background thread.

In [ ]:
import neuroglancer, threading, urllib.parse, re
from cellmap_flow.globals import g
from cellmap_flow.dashboard.app import app
from cellmap_flow.dashboard import state

# Monkey-patch get_raw_layer for remote (http/s3/gs) zarr URLs. The
# upstream implementation calls os.listdir(<url>) to enumerate scale
# levels, which doesn't work over HTTP — it raises, the exception
# handler sets is_multiscale=False, but the path has already been
# stripped to the group URL, so the fallback then opens a zarr Group
# as if it were an Array and crashes on ds.shape. The remote-safe
# fix is to just hand NG the multiscale group URL and let NG's own
# zarr driver pick the right level.
import cellmap_flow.utils.scale_pyramid as _sp
import cellmap_flow.dashboard.routes.pipeline as _pipe
_orig_get_raw_layer = _sp.get_raw_layer
def _remote_safe_get_raw_layer(dataset_path, normalize=True, wrap_raw=True):
    if dataset_path.startswith(("http://", "https://", "s3://", "gs://")):
        group_url = re.sub(r"/s\d+/?$", "", dataset_path)
        return neuroglancer.ImageLayer(source=f"zarr://{group_url}")
    return _orig_get_raw_layer(dataset_path, normalize=normalize, wrap_raw=wrap_raw)
_sp.get_raw_layer = _remote_safe_get_raw_layer
_pipe.get_raw_layer = _remote_safe_get_raw_layer
print("[patch] get_raw_layer remote-URL-safe")

# Bind NG to a known port so we can tunnel it. Must happen BEFORE
# constructing the Viewer (which spins up the server).
neuroglancer.set_server_bind_address("0.0.0.0", bind_port=NG_PORT)

NG_TUNNEL_URL, _ng_tunnel = start_cloudflared(NG_PORT, "neuroglancer")
print(f"NG_TUNNEL_URL = {NG_TUNNEL_URL}")

g.dataset_path = DATASET
viewer = neuroglancer.Viewer()
g.viewer = viewer

# str(viewer) is whatever NG's _get_server_url returned — in Colab
# that's a proxyPort URL we don't want. Replace its origin with the
# cloudflared tunnel; keep the path (which contains the viewer token).
ng_native = str(viewer)
ng_path = urllib.parse.urlparse(ng_native).path or "/"
state.NEUROGLANCER_URL = NG_TUNNEL_URL + ng_path
print(f"NG iframe URL (cloudflared): {state.NEUROGLANCER_URL}")

# Pre-populate g.jobs so the dashboard's /api/process knows about the
# (already running) inference server. host = cloudflared tunnel so
# the NG iframe can reach it without Colab cookies.
class FakeJob:
    def __init__(self, model_name, host):
        self.model_name = model_name
        self.host = host
g.jobs.append(FakeJob(model_name=MODEL_NAME, host=INFERENCE_URL))

# Seed the viewer with raw + inference layers so the iframe shows
# something on first load (before any /api/process click).
with viewer.txn() as s:
    s.dimensions = neuroglancer.CoordinateSpace(
        names=["z", "y", "x"], units="nm", scales=[8, 8, 8],
    )
    raw_url = re.sub(r"/s\d+/?$", "", DATASET)
    s.layers["data"] = neuroglancer.ImageLayer(source=f"zarr://{raw_url}")
    s.layers[MODEL_NAME] = neuroglancer.ImageLayer(
        source=f"zarr://{INFERENCE_URL}/{MODEL_NAME}/",
    )

# Dashboard Flask in a background thread.
dash_thread = threading.Thread(
    target=lambda: app.run(host="0.0.0.0", port=DASHBOARD_PORT,
                            threaded=True, use_reloader=False, debug=False),
    daemon=True,
)
dash_thread.start()
import time as _t; _t.sleep(2)
print(f"dashboard listening on :{DASHBOARD_PORT}")

## 6. Open the dashboard

In [ ]:
from google.colab.output import eval_js
DASHBOARD_URL = eval_js(f"google.colab.kernel.proxyPort({DASHBOARD_PORT})").rstrip("/")
print()
print("=" * 70)
print(f"DASHBOARD URL  (Colab proxyPort — only works in your browser, requires Colab session):")
print(f"  {DASHBOARD_URL}")
print()
print(f"NEUROGLANCER URL  (cloudflared — embedded in dashboard iframe):")
print(f"  {state.NEUROGLANCER_URL}")
print()
print(f"INFERENCE URL  (cloudflared — used by NG to fetch chunks):")
print(f"  {INFERENCE_URL}")
print("=" * 70)

## 7. Keep-alive

Stop with ▢ to tear down. Drains server logs as they come in.

In [ ]:
import time, select

def drain(proc, label):
    while True:
        r, _, _ = select.select([proc.stdout], [], [], 0)
        if not r: return
        line = proc.stdout.readline()
        if not line: return
        print(f"[{label}] {line}", end="")

try:
    while True:
        drain(server, "server")
        if server.poll() is not None:
            drain(server, "server")
            print(f"\n[server] exited rc={server.returncode}.")
            break
        time.sleep(2)
finally:
    try: server.terminate()
    except Exception: pass
